<a href="https://colab.research.google.com/github/naaaddaitorto-star/Lab-4-llm-decision-support/blob/main/lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Naa Adai Torto

**Student ID:** 15022028

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [7]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
from dotenv import load_dotenv
from groq import Groq
load_dotenv()


# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
# from google import genai

# client = genai.Client(api_key = API_KEY)

# interaction = client.interactions.create(
#     model="gemini-3.6-flash",
#     input="Explain how AI works in a few words"
# )
# print(interaction.output_text)

client = Groq(api_key = API_KEY)

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [8]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
      response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
        )
      return response



#
# TODO: Call it once with a simple question and print the answer.
response = ask_llm(user_prompt="Explain how korean drama has such great plots.")
print(response.choices[0].message.content)
# TODO: Print response.usage as well — how many tokens did your call consume?
print(response.usage.completion_tokens)

Korean dramas, also known as K-dramas, have gained worldwide popularity for their engaging storylines, memorable characters, and high production quality. Several factors contribute to their great plots:

1. **Cultural significance and context**: Korean dramas often reflect the country's cultural values, social issues, and historical events. This adds depth and authenticity to the stories, making them relatable and interesting to both domestic and international audiences.
2. **Complex characters and character development**: K-dramas typically feature well-rounded, multi-dimensional characters with rich backstories. The characters' personalities, motivations, and relationships are expertly woven into the narrative, creating a sense of emotional investment in the story.
3. **Tightly written scripts**: Korean drama scripts are often written by experienced writers who have a deep understanding of storytelling and character development. The scripts are carefully crafted to balance action, ro

**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:**
1.A system role is what specifies how the model should behave.eg. You are my Quantitative Methods Lecturer. The user role is the instructions given to the model for it to work on. An example is, give me a detailed explanation on the topic Project Management.

2. The token is the smallest unit of data(text) the model can work on at a time. API providers bill per token to be able to acquire the right cost per every request sent.


### Part 1.2 — Temperature: the randomness dial

In [9]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
print("Temperature: 0.0")
for t in range(5):
  response = ask_llm("Suggest 3 romantic comedy Chinese dramas.", temperature=0.0)
  print(response.choices[0].message.content)
print("Temperature: 1.2")
for t in range(5):
  response = ask_llm("Suggest 3 romantic comedy Chinese dramas.", temperature=1.2)
  print(response.choices[0].message.content)
#

# TODO: Print all 10 answers, grouped by temperature.


Temperature: 0.0
Here are three romantic comedy Chinese dramas that you might enjoy:

1. "Well-Intended Love" (2019) - This drama tells the story of a young woman who pretends to have leukemia in order to get a bone marrow transplant from a wealthy CEO. As they spend more time together, they develop feelings for each other, but their relationship is complicated by their initial deception.

2. "Love O2O" (2016) - This drama follows the story of a college student who becomes involved in an online game and falls in love with a fellow player. When they meet in real life, they must navigate their feelings for each other amidst the challenges of their online and offline relationships.

3. "Go Ahead" (2020) - This drama tells the story of three young people who become like a family to each other after being orphaned at a young age. As they grow up and face various challenges, they must navigate their feelings for each other and figure out what it means to be in love.

All three dramas have a 

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:**

1. So I observed that between the two temperatures used, the 0.0 temperature is low so the variation between the next answers it gives is not vast and it also considers how high the probability is. But for the 1.2 temperature it is high hence when it has to predict the the next possible answer it kind of picks at random and slightly ignores the probabilities.
For the loan management system I think that have a low temperature is best because it first considers or weighs the probabilities of all loan request to see which one is less risky to give the loan. Also,from what I understand, for the loan system, it is better for two officers to get similar predictions because it encourages data consistency. This will help prevent hallucinations by model where false facts could be added to predictions reducing risks.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [10]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [11]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critical
#
# Define SUMMARY_PROMPT_V1 as a function that generates the user prompt string
print("\n--- SUMMARY_PROMPT_V1 Outputs ---")
SUMMARY_PROMPT_V1 = f"Summarize: "
response_v1_l002 = ask_llm(user_prompt=SUMMARY_PROMPT_V1 + LETTERS["L002"])
print(response_v1_l002.choices[0].message.content)

response_v1_l006 = ask_llm(user_prompt=SUMMARY_PROMPT_V1 + LETTERS["L006"])
print(response_v1_l006.choices[0].message.content)


# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

# Define the system prompt for V2
SUMMARY_PROMPT_V2_SYSTEM = "You are an assistant to a microfinance loan officer. Summarize loan applications factually, neutrally, with no invented details, in 3-4 sentences."
def SUMMARY_PROMPT_V2_USER(letter_text):
  return f"Summarize this loan application:\n\n{letter_text}"

print("\n--- SUMMARY_PROMPT_V2 Outputs ---")
# Run V2 on the same two letters at temperature=0
response_v2_l002 = ask_llm(user_prompt=SUMMARY_PROMPT_V2_USER(LETTERS["L002"]),
                           system_prompt=SUMMARY_PROMPT_V2_SYSTEM,
                           temperature=0)
print(response_v2_l002.choices[0].message.content)

response_v2_l006 = ask_llm(user_prompt=SUMMARY_PROMPT_V2_USER(LETTERS["L006"]),
                           system_prompt=SUMMARY_PROMPT_V2_SYSTEM,
                           temperature=0)
print(response_v2_l006.choices[0].message.content)

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.



--- SUMMARY_PROMPT_V1 Outputs ---
Kwame Boateng, a commercial driver in Kumasi, is requesting GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow period in business, but expects it to improve after the festive season. Although he has no collateral, he's asking for urgent assistance and promises to repay the loan as soon as possible.
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience in these ventures, but claims to be "business-minded" based on his friends' opinions. He promises to repay the loan within one year, once his businesses are successful, and offers his trustworthiness as assurance, as he has no collateral to provide.

--- SUMMARY_PROMPT_V2 Outputs ---
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and sett

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:**
1. V1,s output uses the same wording and phrases in the letter which still remains informal for the context, and  an examples is "He needs funds to repair his vehicle's engine and pay off personal debts.". But for V2 output it has paraphrased and used differnt words that give the same message. An example is "He does not have collateral to offer at this time and is seeking a flexible repayment arrangement.". The V2 is also more formal.
2. The "no invented details" is an essential instruction to prevent errors or model failure through hallucination which will ensure that the system gets facts from text provided. This term Hallusination is the fairlure mode in llm where it gives confident, false facts and in this case out of the text. For a loan system if this does not exist false claims that a person has collateral can be given or included in the summary.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [12]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

# def EXTRACT_PROMPT(applicant_name, amount_ghs, purpose, monthly_profit_ghs, has_collateral_or_guarantor, repayment_months):
#   if "If a field is not stated in the letter, use null. Do not guess.":
#     return "null"
#   else:
#     return{
#         applicant_name: "string",
#         amount_ghs: "number",
#         purpose: "string",
#         monthly_profit_ghs: "number or null",
#         has_collateral_or_guarantor: "boolean",
#         repayment_months: "number or null"
#     }

EXTRACT_PROMPT = """
Given the following information about a loan application, do the following:
- Extract the applicant's name as a string (key is applicant_name).
- Extract the amount the person is requesting for (key is amount_ghs).
- Extract the purpose of the loan (key is purpose).
- Extract the monthly profit the person is expecting (key is monthly_profit_ghs).
- Determine if the applicant has collateral or a guarantor as a boolean (key is has_collateral_or_guarantor).
- Extract the number of months the person is expecting to repay the loan (key is repayment_months).

Return this data strictly as a JSON object with no preamble or postamble. As a guide, here is a sample application, and the expected output:
Letter: "Dear Loan Manager, I am Anokor Tetteh, an  apprentice at dark and lovely beauty salon, a student of Abrantie College. I request GHS 10,000 to purchase a kiosk to start my own salon and employ apprentice. Last year I earned a revenue of GHS 13,000; monthly profit averages GHS 1,500 and
I hold a fixed deposit of GHS 2,500 with Republic bank which I can pledge. Proposed repayment: GHS 500 monthly for 12 months. Unfortunately I have no collateral at the moment but God willing everything will be fine."
Output:
{
    "applicant_name": "Anokor Tetteh",
    "amount_ghs": 10000,
    "purpose": "purchase a kiosk to start my own salon and employ apprentice",
    "monthly_profit_ghs": 1500,
    "has_collateral_or_guarantor": false,
    "repayment_months": 12
}

Critical Information:
- The example provided is a an example to guide you in the output format, not the example to run it on.
- If a field is not stated in the letter, use null. Do not guess. Return "null".
- Do not invent any details.

See letter below:
"""


#


# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).
import json

def extract_fields(letter_text, temperature = 0.0):
  response = ask_llm(user_prompt=EXTRACT_PROMPT + letter_text, temperature = temperature)
  if response.choices[0].message.content == "null":
    return None
  else:
    return json.loads(response.choices[0].message.content)

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.
import pandas as pd
info_df = pd.DataFrame(columns=["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs", "has_collateral_or_guarantor", "repayment_months"])

for key, val in LETTERS.items():
  res = extract_fields(val)
  if res == None:
    info_df.loc[key] = [None, None, None, None, None, None]
  else:
    info_df.loc[key] = [res["applicant_name"], res["amount_ghs"], res["purpose"], res["monthly_profit_ghs"], res["has_collateral_or_guarantor"], res["repayment_months"]]



In [13]:
print(info_df)

                          applicant_name  amount_ghs  \
L001                       Akosua Mensah        8000   
L002                       Kwame Boateng       25000   
L003                          Efua Darko       15000   
L004                           Yaw Owusu       12000   
L005  Adenta Women's Weaving Cooperative       30000   
L006                                Kofi       50000   

                                                purpose monthly_profit_ghs  \
L001    buy a deep freezer and expand into frozen foods                900   
L002  repair my trotro engine and settle some person...               None   
L003  purchase two industrial sewing machines and fa...               2800   
L004                        for feed and 500 new layers               1500   
L005  buy a bulk order of yarn directly from the fac...               None   
L006  start a car washing business, a provision shop...               None   

      has_collateral_or_guarantor repayment_months  
L001   

**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:**
1. A few-shot example is different from letters and used to train the model on how to respond and then the six letters are given to the model to test if it has actually learnt.
2. "use null, do not guess"  is used as a precaution to not hallucinate or add its own fact during the information extraction from the six letters. Without the instruction the model might be creative and add false facts during extraction.
3. This is because with temperature = 0, high probabilities are used to predict next output which ensures data consistency. This same temperature is not used for creative task because when being creative random outputs can be put together to be used as the final result but for extraction outputs predicted must match information in the main source.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [14]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

BRIEF_PROMPT = """
After receiving letters and extracting information fromm the JSON, add the following to produce a reccomendation breif do the following:
- List the strengths of each letter using  bullet points to make reading clearly easier.
- Highlight risks in each letter and give clear output using bullet points.
- For missing data if output is null or none or false then it falls under the missing data category.
- Then suggest the next steps from the data given. The next step could be to:
    - call individual for an interview
    - request important document for confirmation
    - flag for senior review.
NOTE: The final decision is made by the human senior officer and not the model.
"""
def recommendation_brief(letter):
  response = ask_llm(user_prompt=BRIEF_PROMPT + letter)
  return response.choices[0].message.content

#
# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
breif = {}
for key, val in LETTERS.items():
  res = recommendation_brief(val)
  breif[key] = res




In [15]:
print("Brief for letter 1.\n")
print(breif["L001"] + "\n\n")

print("Brief for letter 2.")
print(breif["L002"]+ "\n\n")

print("Brief for letter 6.")
print(breif["L006"]+ "\n\n")

print("Brief for letter 3.")
print(breif["L003"]+ "\n\n")

Brief for letter 1.

**Recommendation Brief**

**Letter Summary:**
The letter is from Akosua Mensah, a provision seller at Makola Market, applying for a loan of GHS 8,000 to expand her business into frozen foods.

**Strengths:**
* 12 years of experience selling provisions at Makola Market
* Consistent profit of GHS 900 per month from her current stall
* GHS 2,500 saved with the susu scheme over the past two years with no missed contributions
* A guarantor, her sister, who is a teacher
* Clear repayment plan of GHS 450 monthly over 20 months

**Risks:**
* No clear information on the applicant's credit history
* No collateral provided for the loan
* Dependence on a single guarantor, who may not be able to fulfill their obligations
* Market risks associated with expanding into frozen foods

**Missing Data:**
* Credit history
* Collateral information
* Detailed business plan for the expansion into frozen foods
* Verification of the guarantor's employment and financial stability

**Next Ste

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:**
1. Yes the system was able to identify the right stregths and red flags in the different letters.
2. Practically, simply approving or rejecting just means we are either giving the loan to the individual or not but making it emphasize the different step clearly outlines the full process before final verdict is made.
Ethically, the suggested steps spell out the check that are made to ensure that the individuals credentials are assessed properly and they are eligible for the loan.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:**
a32e3cb


---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [16]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).
fields = ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs", "has_collateral_or_guarantor", "repayment_months"]

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.
def evaluate_fields(prediction, gold):
    if gold is None:
        return prediction is None
    if isinstance(gold, bool):
        return prediction == gold
    if isinstance(gold, (int, float)):
        try:
            return float(prediction) == float(gold)
        except (TypeError, ValueError):
            return False
    if isinstance(gold, str):
        if prediction is None:
            return False
        # Case-insensitive substring/semantic match
        return any(term.lower() in prediction.lower() for term in gold.split() if len(term) > 3)
    return prediction == gold

accurate_matching_rows = []
for field in fields:
    row = {"Field": field}
    matches = []
    for lkey in ["L001", "L003", "L006"]:
        pred_val = info_df.loc[lkey, field]
        gold_val = GOLD[lkey][field]
        is_match = evaluate_fields(pred_val, gold_val)
        row[lkey] = f"Pred: {pred_val} | Gold: {gold_val} ({'PASS' if is_match else 'FAIL'})"
        matches.append(1 if is_match else 0)
    row["Accuracy"] = f"{(sum(matches)/len(matches))*100:.1f}%"
    accurate_matching_rows.append(row)

df_acc_table = pd.DataFrame(accurate_matching_rows).set_index("Field")
display(df_acc_table)



,L001,L003,L006,Accuracy
Field,,,,
applicant_name,Pred: Akosua Mensah | Gold: Akosua Mensah (PASS),Pred: Efua Darko | Gold: Efua Darko (PASS),Pred: Kofi | Gold: Kofi (PASS),100.0%
amount_ghs,Pred: 8000 | Gold: 8000 (PASS),Pred: 15000 | Gold: 15000 (PASS),Pred: 50000 | Gold: 50000 (PASS),100.0%
purpose,Pred: buy a deep freezer and expand into froze...,Pred: purchase two industrial sewing machines ...,"Pred: start a car washing business, a provisio...",100.0%
monthly_profit_ghs,Pred: 900 | Gold: 900 (PASS),Pred: 2800 | Gold: 2800 (PASS),Pred: None | Gold: None (PASS),100.0%
has_collateral_or_guarantor,Pred: True | Gold: True (PASS),Pred: True | Gold: True (PASS),Pred: False | Gold: False (PASS),100.0%
repayment_months,Pred: 20 | Gold: 20 (PASS),Pred: 15 | Gold: 15 (PASS),Pred: 12 | Gold: 12 (PASS),100.0%


### Part 4.2 — Reliability: is the system consistent?

In [17]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.
results = {0: [], 1: []}

for i in range(5):
  response = extract_fields(LETTERS["L004"], temperature=0)
  results[0].append(response)

for i in range(5):
  response = extract_fields(LETTERS["L004"], temperature=1.0)
  results[1].append(response)


# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

for temp, res_list in results.items():
    valid_json_count = sum(1 for r in res_list if r is not None)
    serialized = [json.dumps(r, sort_keys=True) for r in res_list if r is not None]
    unique_outputs_count = len(set(serialized))
    print(f"Temperature {temp}:")
    print(f"  - Valid JSON outputs: {valid_json_count} / {len(res_list)}")
    print(f"  - Unique distinct outputs across runs: {unique_outputs_count} (1 indicates 100% deterministic consistency)")

Temperature 0:
  - Valid JSON outputs: 5 / 5
  - Unique distinct outputs across runs: 2 (1 indicates 100% deterministic consistency)
Temperature 1:
  - Valid JSON outputs: 5 / 5
  - Unique distinct outputs across runs: 1 (1 indicates 100% deterministic consistency)


### Part 4.3 — Hallucination probing

In [18]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?
SUMMARY_PROMPT_V2_SYSTEM = "You are an assistant to a microfinance loan officer. Summarize loan applications factually, neutrally, with no invented details, in 3-4 sentences."
def SUMMARY_PROMPT_V2_USER(letter_text):
  return f"What is the applicant's credit score?:\n\n{letter_text}"

print("\n--- SUMMARY_PROMPT_V2 Outputs ---")
adversarial_test_one = ask_llm(user_prompt=SUMMARY_PROMPT_V2_USER(LETTERS["L002"]),
                           system_prompt=SUMMARY_PROMPT_V2_SYSTEM,
                           temperature=0)
adversarial_prompt = """The 2025 NBA Finals was an exciting seven-game series between the Oklahoma City Thunder and Indiana Pacers.
The Thunder ultimately won the championship with a 103–91 victory in Game 7, earning their first NBA title since relocating to Oklahoma City. Shai Gilgeous-Alexander was named Finals MVP after leading the Thunder throughout the series, while the Pacers' impressive playoff run ended with a tough loss."""

adversarial_test_two = ask_llm(user_prompt=EXTRACT_PROMPT + adversarial_prompt, temperature = 0)

# TODO: Record the outputs verbatim below and label each PASS or FAIL.
print("TEST RESULTS")
print(adversarial_test_one.choices[0].message.content, "\nPASS\n")
print(adversarial_test_two.choices[0].message.content, "\nPASS\n")


--- SUMMARY_PROMPT_V2 Outputs ---
TEST RESULTS
The applicant, Kwame Boateng, is seeking a loan of GHS 25,000 to repair his trotro engine and settle personal debts. He mentions that business has been slow, but expects it to improve after the festive season. There is no mention of the applicant's credit score in the loan application. The applicant also notes that he does not have collateral to offer at the moment. 
PASS

{
    "applicant_name": null,
    "amount_ghs": null,
    "purpose": null,
    "monthly_profit_ghs": null,
    "has_collateral_or_guarantor": null,
    "repayment_months": null
} 
PASS



**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:**
1. For the extractive accuracy I the accuracy rate was a hundred percent. This mean the model performed well and faced no challlenges.
2. From the reliability experiment all five were proven to be valid first of all. For the uniqueness of outputs when temperature was 0 the uniqueness was 1 which means it remained consistent. But for when temperature was set to 1, it produced two unique outputs which means output was not consistent.
3. No, the system did not hallucinate during probing this is shown because for the summarize function it summarized the right text for the corresponding letter and during extraction it retuned null values because the text was irrelevant.

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:**
1. For a bank , first of all workers interact with people of different backgrounds, both educated and uneducated. Knowing these fact when it comes to requesting for loans hindrances like not using the right formal language exist . But in our daily life it said that don't judge book by its cover . So what I think is that yes with the right management the system can be used for analyzing but necessary investigations must be carried out before making final decision. However the languge used must maintain a certain structure to be easily recognized by the model.
2. I think with the issue of privay if data is not handeled well and falls in the hands of irrational people.  However, I think methods like the encryption of data must be ensured. For the API checls must be made to ensure that model does not use personal data as an example for training.
3. The first safegaurd is that though system would be used in day to day processes the final decision making would be done by humans and with good training and management reviews on sytem should be done regularly. The safegaurd is considering appeal processes, if the system recognizes potential eligiblity of people requesting for loans with credible despite grammartical shortcomings, human review is needed after flagging individuals.
The  second safegaurd is to keep log that keep and record prompts given, outputs produced by model and the final decision made by user. This will help ensure  consistency.

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.